In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import KFold, train_test_split
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from deepface import DeepFace
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import gc

In [2]:
DATA_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4"
SEED = 123
all_paths = []
all_labels = []
class_names = ['Autistic', 'Non_Autistic']

IMG_SIZE = (224, 224)
BATCH_SIZE = 8          
SEED = 123
fold_no = 1              
fold_results = []        
num_folds = 5

methana idn oni na aith

In [3]:
for class_idx, class_name in enumerate(class_names):
    class_folder = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_folder):
        all_paths.append(os.path.join(class_folder, img_name))
        # Logic: Autistic is class_idx 0. 
        # We will apply (1 - y) later in the pipeline to make Autistic = 1
        all_labels.append(class_idx)

In [4]:
all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

In [5]:
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.10, random_state=SEED, stratify=all_labels
)

In [6]:
print(f"Permanent Test Set isolated: {len(test_paths)} images.")
print(f"Data remaining for K-Fold: {len(train_val_paths)} images.")

Permanent Test Set isolated: 392 images.
Data remaining for K-Fold: 3525 images.


In [7]:
# --- 4. Define K-Fold ---
num_folds = 5
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)

fold_no = 1
# This list will store the reliability metrics for each fold
fold_results = []


In [8]:
FOLDS_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\Folds"
if not os.path.exists(FOLDS_DIR):
    os.makedirs(FOLDS_DIR)
    print(f"Created directory: {FOLDS_DIR}")

np.save(os.path.join(FOLDS_DIR, 'permanent_test_paths.npy'), test_paths)
np.save(os.path.join(FOLDS_DIR, 'permanent_test_labels.npy'), test_labels)
print(f"Permanent Test Set locked to disk: {len(test_paths)} images.")

splits = list(kfold.split(train_val_paths, train_val_labels))

for i, (train_index, val_index) in enumerate(splits):
    fold_id = i + 1
    
    # Extract the paths and labels for this specific fold
    train_paths = train_val_paths[train_index]
    train_labels = train_val_labels[train_index]
    val_paths = train_val_paths[val_index]
    val_labels = train_val_labels[val_index]
    
    # Save to disk
    np.save(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_paths.npy'), train_paths)
    np.save(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_labels.npy'), train_labels)
    np.save(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_paths.npy'), val_paths)
    np.save(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_labels.npy'), val_labels)
    
    print(f"Fold {fold_id} saved: {len(train_paths)} train, {len(val_paths)} val images.")

print("\nAll folds are successfully locked to the hard drive.")

Permanent Test Set locked to disk: 392 images.
Fold 1 saved: 2820 train, 705 val images.
Fold 2 saved: 2820 train, 705 val images.
Fold 3 saved: 2820 train, 705 val images.
Fold 4 saved: 2820 train, 705 val images.
Fold 5 saved: 2820 train, 705 val images.

All folds are successfully locked to the hard drive.


Oni na me kalla

In [4]:
FOLDS_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\Folds"
CURRENT_FOLD = 5
train_paths_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{CURRENT_FOLD}_train_paths.npy'), allow_pickle=True)
train_labels_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{CURRENT_FOLD}_train_labels.npy'), allow_pickle=True)
val_paths_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{CURRENT_FOLD}_val_paths.npy'), allow_pickle=True)
val_labels_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{CURRENT_FOLD}_val_labels.npy'), allow_pickle=True)
print(f"--- FOLD {CURRENT_FOLD} LOADED ---")
print(f"Training images: {len(train_paths_fold)}")
print(f"Validation images: {len(val_paths_fold)}")

--- FOLD 5 LOADED ---
Training images: 2820
Validation images: 705


In [5]:
# --- 3. DATA PIPELINE ---
data_augmentation_layers = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03), 
    tf.keras.layers.RandomContrast(0.1),
])

def process_fold_data(file_path, label, augment=False):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    
    # VGG-Face Preprocessing: BGR conversion and Mean Subtraction
    img = img[..., ::-1] # RGB to BGR
    mean = [93.5940, 104.7624, 129.1863]
    img = img - mean

    if augment:
        img = tf.expand_dims(img, 0)
        img = data_augmentation_layers(img, training=True)
        img = tf.squeeze(img, 0)

    # Class logic: Maps to Autistic=1, Non_Autistic=0
    target_label = 1.0 - tf.cast(label, tf.float32)
    return img, {
        'classification_output': target_label, 
        'asd_feature_vector': target_label
    }

train_ds_fold = (tf.data.Dataset.from_tensor_slices((train_paths_fold, train_labels_fold))
                 .shuffle(len(train_paths_fold))
                 .map(lambda x, y: process_fold_data(x, y, augment=True), num_parallel_calls=tf.data.AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(tf.data.AUTOTUNE))

val_ds_fold = (tf.data.Dataset.from_tensor_slices((val_paths_fold, val_labels_fold))
               .map(lambda x, y: process_fold_data(x, y, augment=False), num_parallel_calls=tf.data.AUTOTUNE)
               .batch(BATCH_SIZE)
               .prefetch(tf.data.AUTOTUNE))

# --- 4. MODEL ARCHITECTURE ---
vgg_model_wrapper = DeepFace.build_model("VGG-Face")
full_vgg_face = vgg_model_wrapper.model 
backbone_output = full_vgg_face.get_layer("conv2d_14").output
feature_extractor = Model(inputs=full_vgg_face.input, outputs=backbone_output)
feature_extractor.trainable = False 

inputs = Input(shape=(224, 224, 3), name="face_input")
x = feature_extractor(inputs, training=False)
x = GlobalAveragePooling2D()(x)
feature_layer = Dense(256, activation='relu', kernel_regularizer=l2(0.01), name='asd_feature_vector')(x)
x = Dropout(0.5)(feature_layer)
prediction = Dense(1, activation='sigmoid', name='classification_output')(x)

unified_model = Model(inputs=inputs, outputs=[prediction, feature_layer])

# --- 5. TRAINING PHASE 1 (FROZEN) ---
lr_scheduler = ReduceLROnPlateau(monitor='val_classification_output_loss', factor=0.5, patience=3, verbose=1)
early_stop = EarlyStopping(monitor='val_classification_output_loss', patience=8, restore_best_weights=True)
checkpoint = ModelCheckpoint(f'fold_{CURRENT_FOLD}_best.h5', monitor='val_classification_output_accuracy', save_best_only=True, mode='max', verbose=1)

unified_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss={'classification_output': 'binary_crossentropy', 'asd_feature_vector': None},
    metrics={'classification_output': ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]}
)

print("\nStarting Phase 1 (Frozen Backbone)...")
unified_model.fit(train_ds_fold, validation_data=val_ds_fold, epochs=30, callbacks=[lr_scheduler, early_stop, checkpoint])

# --- 6. TRAINING PHASE 2 (FINE-TUNING) ---
feature_extractor.trainable = True

for layer in feature_extractor.layers[:-12]:
    layer.trainable = False

# Re-compile is required after changing trainable status
unified_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss={'classification_output': 'binary_crossentropy', 'asd_feature_vector': None},
    metrics={'classification_output': ['accuracy']}
)

print("\nStarting Phase 2 (Fine-Tuning Last 12 Layers)...")
history_p2 = unified_model.fit(train_ds_fold, validation_data=val_ds_fold, epochs=15, callbacks=[lr_scheduler, early_stop, checkpoint])

# --- 7. FINAL LOGGING ---
final_acc = max(history_p2.history['val_classification_output_accuracy'])
print(f"\nFINISHED FOLD {CURRENT_FOLD}")
print(f"Peak Validation Accuracy: {final_acc:.4f}")


Starting Phase 1 (Frozen Backbone)...
Epoch 1/30
353/353 [==============================] - ETA: 0s - loss: 4.5838 - classification_output_loss: 0.8925 - classification_output_accuracy: 0.6238 - classification_output_precision: 0.6206 - classification_output_recall: 0.6272
Epoch 1: val_classification_output_accuracy improved from -inf to 0.65248, saving model to fold_5_best.h5
353/353 [==============================] - 78s 196ms/step - loss: 4.5838 - classification_output_loss: 0.8925 - classification_output_accuracy: 0.6238 - classification_output_precision: 0.6206 - classification_output_recall: 0.6272 - val_loss: 3.9731 - val_classification_output_loss: 0.7744 - val_classification_output_accuracy: 0.6525 - val_classification_output_precision: 0.7137 - val_classification_output_recall: 0.5141 - lr: 1.0000e-04
Epoch 2/30
353/353 [==============================] - ETA: 0s - loss: 3.4321 - classification_output_loss: 0.5061 - classification_output_accuracy: 0.7582 - classification_outp

In [ ]:
Starting Phase 1 (Frozen Backbone)...
Epoch 1/30
353/353 [==============================] - ETA: 0s - loss: 4.4780 - classification_output_loss: 0.8522 - classification_output_accuracy: 0.6348 - classification_output_precision: 0.6304 - classification_output_recall: 0.6439
Epoch 1: val_classification_output_accuracy improved from -inf to 0.66667, saving model to fold_1_best.h5
353/353 [==============================] - 106s 257ms/step - loss: 4.4780 - classification_output_loss: 0.8522 - classification_output_accuracy: 0.6348 - classification_output_precision: 0.6304 - classification_output_recall: 0.6439 - val_loss: 3.8227 - val_classification_output_loss: 0.7319 - val_classification_output_accuracy: 0.6667 - val_classification_output_precision: 0.7479 - val_classification_output_recall: 0.5042 - lr: 1.0000e-04
Epoch 2/30
353/353 [==============================] - ETA: 0s - loss: 3.3183 - classification_output_loss: 0.5013 - classification_output_accuracy: 0.7635 - classification_output_precision: 0.7572 - classification_output_recall: 0.7728
Epoch 2: val_classification_output_accuracy improved from 0.66667 to 0.68369, saving model to fold_1_best.h5
353/353 [==============================] - 63s 178ms/step - loss: 3.3183 - classification_output_loss: 0.5013 - classification_output_accuracy: 0.7635 - classification_output_precision: 0.7572 - classification_output_recall: 0.7728 - val_loss: 3.3194 - val_classification_output_loss: 0.7422 - val_classification_output_accuracy: 0.6837 - val_classification_output_precision: 0.8037 - val_classification_output_recall: 0.4873 - lr: 1.0000e-04
Epoch 3/30
353/353 [==============================] - ETA: 0s - loss: 2.7831 - classification_output_loss: 0.4011 - classification_output_accuracy: 0.8255 - classification_output_precision: 0.8189 - classification_output_recall: 0.8340
Epoch 3: val_classification_output_accuracy improved from 0.68369 to 0.70355, saving model to fold_1_best.h5
353/353 [==============================] - 57s 162ms/step - loss: 2.7831 - classification_output_loss: 0.4011 - classification_output_accuracy: 0.8255 - classification_output_precision: 0.8189 - classification_output_recall: 0.8340 - val_loss: 2.9180 - val_classification_output_loss: 0.7184 - val_classification_output_accuracy: 0.7035 - val_classification_output_precision: 0.7903 - val_classification_output_recall: 0.5552 - lr: 1.0000e-04
Epoch 4/30
353/353 [==============================] - ETA: 0s - loss: 2.3629 - classification_output_loss: 0.3216 - classification_output_accuracy: 0.8688 - classification_output_precision: 0.8605 - classification_output_recall: 0.8789
Epoch 4: val_classification_output_accuracy did not improve from 0.70355
353/353 [==============================] - 58s 164ms/step - loss: 2.3629 - classification_output_loss: 0.3216 - classification_output_accuracy: 0.8688 - classification_output_precision: 0.8605 - classification_output_recall: 0.8789 - val_loss: 2.7685 - val_classification_output_loss: 0.8772 - val_classification_output_accuracy: 0.6695 - val_classification_output_precision: 0.8371 - val_classification_output_recall: 0.4221 - lr: 1.0000e-04
Epoch 5/30
353/353 [==============================] - ETA: 0s - loss: 2.0449 - classification_output_loss: 0.2853 - classification_output_accuracy: 0.8897 - classification_output_precision: 0.8857 - classification_output_recall: 0.8939
Epoch 5: val_classification_output_accuracy did not improve from 0.70355
353/353 [==============================] - 60s 170ms/step - loss: 2.0449 - classification_output_loss: 0.2853 - classification_output_accuracy: 0.8897 - classification_output_precision: 0.8857 - classification_output_recall: 0.8939 - val_loss: 2.5440 - val_classification_output_loss: 0.9100 - val_classification_output_accuracy: 0.6709 - val_classification_output_precision: 0.8040 - val_classification_output_recall: 0.4533 - lr: 1.0000e-04
Epoch 6/30
353/353 [==============================] - ETA: 0s - loss: 1.7636 - classification_output_loss: 0.2417 - classification_output_accuracy: 0.9060 - classification_output_precision: 0.9025 - classification_output_recall: 0.9095
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 6: val_classification_output_accuracy did not improve from 0.70355
353/353 [==============================] - 124s 352ms/step - loss: 1.7636 - classification_output_loss: 0.2417 - classification_output_accuracy: 0.9060 - classification_output_precision: 0.9025 - classification_output_recall: 0.9095 - val_loss: 2.2302 - val_classification_output_loss: 0.8145 - val_classification_output_accuracy: 0.6965 - val_classification_output_precision: 0.7932 - val_classification_output_recall: 0.5326 - lr: 1.0000e-04
Epoch 7/30
353/353 [==============================] - ETA: 0s - loss: 1.5679 - classification_output_loss: 0.2033 - classification_output_accuracy: 0.9355 - classification_output_precision: 0.9279 - classification_output_recall: 0.9437
Epoch 7: val_classification_output_accuracy did not improve from 0.70355
353/353 [==============================] - 59s 168ms/step - loss: 1.5679 - classification_output_loss: 0.2033 - classification_output_accuracy: 0.9355 - classification_output_precision: 0.9279 - classification_output_recall: 0.9437 - val_loss: 2.1871 - val_classification_output_loss: 0.8739 - val_classification_output_accuracy: 0.6894 - val_classification_output_precision: 0.8073 - val_classification_output_recall: 0.4986 - lr: 5.0000e-05
Epoch 8/30
353/353 [==============================] - ETA: 0s - loss: 1.4468 - classification_output_loss: 0.1836 - classification_output_accuracy: 0.9404 - classification_output_precision: 0.9340 - classification_output_recall: 0.9473
Epoch 8: val_classification_output_accuracy improved from 0.70355 to 0.70496, saving model to fold_1_best.h5
353/353 [==============================] - 57s 162ms/step - loss: 1.4468 - classification_output_loss: 0.1836 - classification_output_accuracy: 0.9404 - classification_output_precision: 0.9340 - classification_output_recall: 0.9473 - val_loss: 2.0673 - val_classification_output_loss: 0.8534 - val_classification_output_accuracy: 0.7050 - val_classification_output_precision: 0.8194 - val_classification_output_recall: 0.5269 - lr: 5.0000e-05
Epoch 9/30
353/353 [==============================] - ETA: 0s - loss: 1.3419 - classification_output_loss: 0.1747 - classification_output_accuracy: 0.9411 - classification_output_precision: 0.9335 - classification_output_recall: 0.9494
Epoch 9: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 9: val_classification_output_accuracy did not improve from 0.70496
353/353 [==============================] - 58s 164ms/step - loss: 1.3419 - classification_output_loss: 0.1747 - classification_output_accuracy: 0.9411 - classification_output_precision: 0.9335 - classification_output_recall: 0.9494 - val_loss: 1.9680 - val_classification_output_loss: 0.8471 - val_classification_output_accuracy: 0.6950 - val_classification_output_precision: 0.7805 - val_classification_output_recall: 0.5439 - lr: 5.0000e-05
Epoch 10/30
353/353 [==============================] - ETA: 0s - loss: 1.2515 - classification_output_loss: 0.1537 - classification_output_accuracy: 0.9557 - classification_output_precision: 0.9532 - classification_output_recall: 0.9580
Epoch 10: val_classification_output_accuracy did not improve from 0.70496
353/353 [==============================] - 58s 164ms/step - loss: 1.2515 - classification_output_loss: 0.1537 - classification_output_accuracy: 0.9557 - classification_output_precision: 0.9532 - classification_output_recall: 0.9580 - val_loss: 1.9544 - val_classification_output_loss: 0.8799 - val_classification_output_accuracy: 0.6965 - val_classification_output_precision: 0.7983 - val_classification_output_recall: 0.5269 - lr: 2.5000e-05
Epoch 11/30
353/353 [==============================] - ETA: 0s - loss: 1.1981 - classification_output_loss: 0.1469 - classification_output_accuracy: 0.9589 - classification_output_precision: 0.9491 - classification_output_recall: 0.9694
Epoch 11: val_classification_output_accuracy did not improve from 0.70496
353/353 [==============================] - 57s 163ms/step - loss: 1.1981 - classification_output_loss: 0.1469 - classification_output_accuracy: 0.9589 - classification_output_precision: 0.9491 - classification_output_recall: 0.9694 - val_loss: 1.8854 - val_classification_output_loss: 0.8572 - val_classification_output_accuracy: 0.6979 - val_classification_output_precision: 0.7846 - val_classification_output_recall: 0.5467 - lr: 2.5000e-05

Starting Phase 2 (Fine-Tuning Last 12 Layers)...
Epoch 1/15
353/353 [==============================] - ETA: 0s - loss: 2.4827 - classification_output_loss: 0.2925 - classification_output_accuracy: 0.8784
Epoch 1: val_classification_output_accuracy did not improve from 0.70496
353/353 [==============================] - 81s 224ms/step - loss: 2.4827 - classification_output_loss: 0.2925 - classification_output_accuracy: 0.8784 - val_loss: 3.0011 - val_classification_output_loss: 0.8161 - val_classification_output_accuracy: 0.6851 - lr: 1.0000e-06
Epoch 2/15
353/353 [==============================] - ETA: 0s - loss: 2.4280 - classification_output_loss: 0.2469 - classification_output_accuracy: 0.9035
Epoch 2: val_classification_output_accuracy improved from 0.70496 to 0.70780, saving model to fold_1_best.h5
353/353 [==============================] - 91s 257ms/step - loss: 2.4280 - classification_output_loss: 0.2469 - classification_output_accuracy: 0.9035 - val_loss: 2.9779 - val_classification_output_loss: 0.8005 - val_classification_output_accuracy: 0.7078 - lr: 1.0000e-06
Epoch 3/15
353/353 [==============================] - ETA: 0s - loss: 2.3941 - classification_output_loss: 0.2200 - classification_output_accuracy: 0.9149
Epoch 3: val_classification_output_accuracy did not improve from 0.70780
353/353 [==============================] - 87s 247ms/step - loss: 2.3941 - classification_output_loss: 0.2200 - classification_output_accuracy: 0.9149 - val_loss: 3.0431 - val_classification_output_loss: 0.8724 - val_classification_output_accuracy: 0.7007 - lr: 1.0000e-06
Epoch 4/15
353/353 [==============================] - ETA: 0s - loss: 2.3559 - classification_output_loss: 0.1884 - classification_output_accuracy: 0.9284
Epoch 4: val_classification_output_accuracy did not improve from 0.70780
353/353 [==============================] - 80s 227ms/step - loss: 2.3559 - classification_output_loss: 0.1884 - classification_output_accuracy: 0.9284 - val_loss: 3.0429 - val_classification_output_loss: 0.8786 - val_classification_output_accuracy: 0.7078 - lr: 1.0000e-06
Epoch 5/15
353/353 [==============================] - ETA: 0s - loss: 2.3206 - classification_output_loss: 0.1595 - classification_output_accuracy: 0.9443
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999987376214e-07.

Epoch 5: val_classification_output_accuracy did not improve from 0.70780
353/353 [==============================] - 77s 219ms/step - loss: 2.3206 - classification_output_loss: 0.1595 - classification_output_accuracy: 0.9443 - val_loss: 3.1039 - val_classification_output_loss: 0.9460 - val_classification_output_accuracy: 0.7050 - lr: 1.0000e-06
Epoch 6/15
353/353 [==============================] - ETA: 0s - loss: 2.2961 - classification_output_loss: 0.1398 - classification_output_accuracy: 0.9528
Epoch 6: val_classification_output_accuracy improved from 0.70780 to 0.71206, saving model to fold_1_best.h5
353/353 [==============================] - 81s 229ms/step - loss: 2.2961 - classification_output_loss: 0.1398 - classification_output_accuracy: 0.9528 - val_loss: 3.1081 - val_classification_output_loss: 0.9533 - val_classification_output_accuracy: 0.7121 - lr: 5.0000e-07
Epoch 7/15
353/353 [==============================] - ETA: 0s - loss: 2.2723 - classification_output_loss: 0.1192 - classification_output_accuracy: 0.9638
Epoch 7: val_classification_output_accuracy improved from 0.71206 to 0.71489, saving model to fold_1_best.h5
353/353 [==============================] - 80s 225ms/step - loss: 2.2723 - classification_output_loss: 0.1192 - classification_output_accuracy: 0.9638 - val_loss: 3.1081 - val_classification_output_loss: 0.9566 - val_classification_output_accuracy: 0.7149 - lr: 5.0000e-07
Epoch 8/15
353/353 [==============================] - ETA: 0s - loss: 2.2699 - classification_output_loss: 0.1201 - classification_output_accuracy: 0.9592
Epoch 8: ReduceLROnPlateau reducing learning rate to 2.499999993688107e-07.

Epoch 8: val_classification_output_accuracy did not improve from 0.71489
353/353 [==============================] - 77s 219ms/step - loss: 2.2699 - classification_output_loss: 0.1201 - classification_output_accuracy: 0.9592 - val_loss: 3.1490 - val_classification_output_loss: 1.0008 - val_classification_output_accuracy: 0.7149 - lr: 5.0000e-07
Epoch 9/15
353/353 [==============================] - ETA: 0s - loss: 2.2532 - classification_output_loss: 0.1058 - classification_output_accuracy: 0.9684
Epoch 9: val_classification_output_accuracy improved from 0.71489 to 0.71631, saving model to fold_1_best.h5
353/353 [==============================] - 80s 228ms/step - loss: 2.2532 - classification_output_loss: 0.1058 - classification_output_accuracy: 0.9684 - val_loss: 3.1229 - val_classification_output_loss: 0.9764 - val_classification_output_accuracy: 0.7163 - lr: 2.5000e-07
Epoch 10/15
353/353 [==============================] - ETA: 0s - loss: 2.2492 - classification_output_loss: 0.1035 - classification_output_accuracy: 0.9695
Epoch 10: val_classification_output_accuracy did not improve from 0.71631
353/353 [==============================] - 78s 220ms/step - loss: 2.2492 - classification_output_loss: 0.1035 - classification_output_accuracy: 0.9695 - val_loss: 3.1514 - val_classification_output_loss: 1.0065 - val_classification_output_accuracy: 0.7121 - lr: 2.5000e-07

FINISHED FOLD 1
Peak Validation Accuracy: 0.7163

In [ ]:
Starting Phase 1 (Frozen Backbone)...
Epoch 1/30
353/353 [==============================] - ETA: 0s - loss: 4.3735 - classification_output_loss: 0.8197 - classification_output_accuracy: 0.6394 - classification_output_precision: 0.6340 - classification_output_recall: 0.6543
Epoch 1: val_classification_output_accuracy improved from -inf to 0.68511, saving model to fold_4_best.h5
353/353 [==============================] - 72s 178ms/step - loss: 4.3735 - classification_output_loss: 0.8197 - classification_output_accuracy: 0.6394 - classification_output_precision: 0.6340 - classification_output_recall: 0.6543 - val_loss: 3.6735 - val_classification_output_loss: 0.6943 - val_classification_output_accuracy: 0.6851 - val_classification_output_precision: 0.8342 - val_classification_output_recall: 0.4587 - lr: 1.0000e-04
Epoch 2/30
353/353 [==============================] - ETA: 0s - loss: 3.1936 - classification_output_loss: 0.5058 - classification_output_accuracy: 0.7557 - classification_output_precision: 0.7534 - classification_output_recall: 0.7582
Epoch 2: val_classification_output_accuracy improved from 0.68511 to 0.72482, saving model to fold_4_best.h5
353/353 [==============================] - 63s 179ms/step - loss: 3.1936 - classification_output_loss: 0.5058 - classification_output_accuracy: 0.7557 - classification_output_precision: 0.7534 - classification_output_recall: 0.7582 - val_loss: 3.1220 - val_classification_output_loss: 0.6851 - val_classification_output_accuracy: 0.7248 - val_classification_output_precision: 0.9110 - val_classification_output_recall: 0.4957 - lr: 1.0000e-04
Epoch 3/30
353/353 [==============================] - ETA: 0s - loss: 2.6393 - classification_output_loss: 0.4017 - classification_output_accuracy: 0.8156 - classification_output_precision: 0.8094 - classification_output_recall: 0.8243
Epoch 3: val_classification_output_accuracy improved from 0.72482 to 0.75319, saving model to fold_4_best.h5
353/353 [==============================] - 58s 164ms/step - loss: 2.6393 - classification_output_loss: 0.4017 - classification_output_accuracy: 0.8156 - classification_output_precision: 0.8094 - classification_output_recall: 0.8243 - val_loss: 2.6630 - val_classification_output_loss: 0.6098 - val_classification_output_accuracy: 0.7532 - val_classification_output_precision: 0.8498 - val_classification_output_recall: 0.6125 - lr: 1.0000e-04
Epoch 4/30
353/353 [==============================] - ETA: 0s - loss: 2.2366 - classification_output_loss: 0.3403 - classification_output_accuracy: 0.8535 - classification_output_precision: 0.8446 - classification_output_recall: 0.8656
Epoch 4: val_classification_output_accuracy did not improve from 0.75319
353/353 [==============================] - 59s 167ms/step - loss: 2.2366 - classification_output_loss: 0.3403 - classification_output_accuracy: 0.8535 - classification_output_precision: 0.8446 - classification_output_recall: 0.8656 - val_loss: 2.4385 - val_classification_output_loss: 0.6901 - val_classification_output_accuracy: 0.7362 - val_classification_output_precision: 0.8571 - val_classification_output_recall: 0.5641 - lr: 1.0000e-04
Epoch 5/30
353/353 [==============================] - ETA: 0s - loss: 1.9149 - classification_output_loss: 0.2942 - classification_output_accuracy: 0.8794 - classification_output_precision: 0.8727 - classification_output_recall: 0.8876
Epoch 5: val_classification_output_accuracy did not improve from 0.75319
353/353 [==============================] - 59s 166ms/step - loss: 1.9149 - classification_output_loss: 0.2942 - classification_output_accuracy: 0.8794 - classification_output_precision: 0.8727 - classification_output_recall: 0.8876 - val_loss: 2.2116 - val_classification_output_loss: 0.7109 - val_classification_output_accuracy: 0.7433 - val_classification_output_precision: 0.8829 - val_classification_output_recall: 0.5584 - lr: 1.0000e-04
Epoch 6/30
353/353 [==============================] - ETA: 0s - loss: 1.6544 - classification_output_loss: 0.2585 - classification_output_accuracy: 0.9032 - classification_output_precision: 0.9015 - classification_output_recall: 0.9047
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 6: val_classification_output_accuracy did not improve from 0.75319
353/353 [==============================] - 58s 164ms/step - loss: 1.6544 - classification_output_loss: 0.2585 - classification_output_accuracy: 0.9032 - classification_output_precision: 0.9015 - classification_output_recall: 0.9047 - val_loss: 2.1164 - val_classification_output_loss: 0.8196 - val_classification_output_accuracy: 0.7305 - val_classification_output_precision: 0.8889 - val_classification_output_recall: 0.5242 - lr: 1.0000e-04
Epoch 7/30
353/353 [==============================] - ETA: 0s - loss: 1.4747 - classification_output_loss: 0.2260 - classification_output_accuracy: 0.9177 - classification_output_precision: 0.9082 - classification_output_recall: 0.9289
Epoch 7: val_classification_output_accuracy improved from 0.75319 to 0.75461, saving model to fold_4_best.h5
353/353 [==============================] - 58s 165ms/step - loss: 1.4747 - classification_output_loss: 0.2260 - classification_output_accuracy: 0.9177 - classification_output_precision: 0.9082 - classification_output_recall: 0.9289 - val_loss: 1.9218 - val_classification_output_loss: 0.7205 - val_classification_output_accuracy: 0.7546 - val_classification_output_precision: 0.8771 - val_classification_output_recall: 0.5897 - lr: 5.0000e-05
Epoch 8/30
353/353 [==============================] - ETA: 0s - loss: 1.3592 - classification_output_loss: 0.2033 - classification_output_accuracy: 0.9351 - classification_output_precision: 0.9303 - classification_output_recall: 0.9403
Epoch 8: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 57s 160ms/step - loss: 1.3592 - classification_output_loss: 0.2033 - classification_output_accuracy: 0.9351 - classification_output_precision: 0.9303 - classification_output_recall: 0.9403 - val_loss: 1.8325 - val_classification_output_loss: 0.7212 - val_classification_output_accuracy: 0.7546 - val_classification_output_precision: 0.8803 - val_classification_output_recall: 0.5869 - lr: 5.0000e-05
Epoch 9/30
353/353 [==============================] - ETA: 0s - loss: 1.2490 - classification_output_loss: 0.1806 - classification_output_accuracy: 0.9461 - classification_output_precision: 0.9403 - classification_output_recall: 0.9523
Epoch 9: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 9: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 57s 162ms/step - loss: 1.2490 - classification_output_loss: 0.1806 - classification_output_accuracy: 0.9461 - classification_output_precision: 0.9403 - classification_output_recall: 0.9523 - val_loss: 1.7830 - val_classification_output_loss: 0.7568 - val_classification_output_accuracy: 0.7518 - val_classification_output_precision: 0.8729 - val_classification_output_recall: 0.5869 - lr: 5.0000e-05
Epoch 10/30
353/353 [==============================] - ETA: 0s - loss: 1.1661 - classification_output_loss: 0.1608 - classification_output_accuracy: 0.9585 - classification_output_precision: 0.9536 - classification_output_recall: 0.9637
Epoch 10: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 59s 168ms/step - loss: 1.1661 - classification_output_loss: 0.1608 - classification_output_accuracy: 0.9585 - classification_output_precision: 0.9536 - classification_output_recall: 0.9637 - val_loss: 1.7528 - val_classification_output_loss: 0.7688 - val_classification_output_accuracy: 0.7546 - val_classification_output_precision: 0.8803 - val_classification_output_recall: 0.5869 - lr: 2.5000e-05
Epoch 11/30
353/353 [==============================] - ETA: 0s - loss: 1.1244 - classification_output_loss: 0.1612 - classification_output_accuracy: 0.9557 - classification_output_precision: 0.9457 - classification_output_recall: 0.9666
Epoch 11: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 59s 167ms/step - loss: 1.1244 - classification_output_loss: 0.1612 - classification_output_accuracy: 0.9557 - classification_output_precision: 0.9457 - classification_output_recall: 0.9666 - val_loss: 1.7221 - val_classification_output_loss: 0.7799 - val_classification_output_accuracy: 0.7532 - val_classification_output_precision: 0.8766 - val_classification_output_recall: 0.5869 - lr: 2.5000e-05

Starting Phase 2 (Fine-Tuning Last 12 Layers)...
Epoch 1/15
353/353 [==============================] - ETA: 0s - loss: 2.3536 - classification_output_loss: 0.3084 - classification_output_accuracy: 0.8649
Epoch 1: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 80s 223ms/step - loss: 2.3536 - classification_output_loss: 0.3084 - classification_output_accuracy: 0.8649 - val_loss: 2.7066 - val_classification_output_loss: 0.6661 - val_classification_output_accuracy: 0.7489 - lr: 1.0000e-06
Epoch 2/15
353/353 [==============================] - ETA: 0s - loss: 2.3051 - classification_output_loss: 0.2680 - classification_output_accuracy: 0.8968
Epoch 2: val_classification_output_accuracy did not improve from 0.75461
353/353 [==============================] - 79s 223ms/step - loss: 2.3051 - classification_output_loss: 0.2680 - classification_output_accuracy: 0.8968 - val_loss: 2.7028 - val_classification_output_loss: 0.6689 - val_classification_output_accuracy: 0.7532 - lr: 1.0000e-06
Epoch 3/15
353/353 [==============================] - ETA: 0s - loss: 2.2636 - classification_output_loss: 0.2326 - classification_output_accuracy: 0.9106
Epoch 3: val_classification_output_accuracy improved from 0.75461 to 0.76454, saving model to fold_4_best.h5
353/353 [==============================] - 472s 1s/step - loss: 2.2636 - classification_output_loss: 0.2326 - classification_output_accuracy: 0.9106 - val_loss: 2.6875 - val_classification_output_loss: 0.6594 - val_classification_output_accuracy: 0.7645 - lr: 1.0000e-06
Epoch 4/15
353/353 [==============================] - ETA: 0s - loss: 2.2283 - classification_output_loss: 0.2030 - classification_output_accuracy: 0.9227
Epoch 4: val_classification_output_accuracy did not improve from 0.76454
353/353 [==============================] - 889s 3s/step - loss: 2.2283 - classification_output_loss: 0.2030 - classification_output_accuracy: 0.9227 - val_loss: 2.7689 - val_classification_output_loss: 0.7463 - val_classification_output_accuracy: 0.7603 - lr: 1.0000e-06
Epoch 5/15
353/353 [==============================] - ETA: 0s - loss: 2.1882 - classification_output_loss: 0.1683 - classification_output_accuracy: 0.9429
Epoch 5: val_classification_output_accuracy improved from 0.76454 to 0.77021, saving model to fold_4_best.h5
353/353 [==============================] - 372s 1s/step - loss: 2.1882 - classification_output_loss: 0.1683 - classification_output_accuracy: 0.9429 - val_loss: 2.7641 - val_classification_output_loss: 0.7470 - val_classification_output_accuracy: 0.7702 - lr: 1.0000e-06
Epoch 6/15
353/353 [==============================] - ETA: 0s - loss: 2.1631 - classification_output_loss: 0.1487 - classification_output_accuracy: 0.9475
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.999999987376214e-07.

Epoch 6: val_classification_output_accuracy did not improve from 0.77021
353/353 [==============================] - 77s 218ms/step - loss: 2.1631 - classification_output_loss: 0.1487 - classification_output_accuracy: 0.9475 - val_loss: 2.8121 - val_classification_output_loss: 0.8004 - val_classification_output_accuracy: 0.7589 - lr: 1.0000e-06
Epoch 7/15
353/353 [==============================] - ETA: 0s - loss: 2.1403 - classification_output_loss: 0.1300 - classification_output_accuracy: 0.9582
Epoch 7: val_classification_output_accuracy did not improve from 0.77021
353/353 [==============================] - 77s 219ms/step - loss: 2.1403 - classification_output_loss: 0.1300 - classification_output_accuracy: 0.9582 - val_loss: 2.8383 - val_classification_output_loss: 0.8293 - val_classification_output_accuracy: 0.7645 - lr: 5.0000e-07
Epoch 8/15
353/353 [==============================] - ETA: 0s - loss: 2.1246 - classification_output_loss: 0.1170 - classification_output_accuracy: 0.9621
Epoch 8: val_classification_output_accuracy improved from 0.77021 to 0.77163, saving model to fold_4_best.h5
353/353 [==============================] - 79s 225ms/step - loss: 2.1246 - classification_output_loss: 0.1170 - classification_output_accuracy: 0.9621 - val_loss: 2.8239 - val_classification_output_loss: 0.8177 - val_classification_output_accuracy: 0.7716 - lr: 5.0000e-07
Epoch 9/15
353/353 [==============================] - ETA: 0s - loss: 2.1076 - classification_output_loss: 0.1029 - classification_output_accuracy: 0.9688
Epoch 9: ReduceLROnPlateau reducing learning rate to 2.499999993688107e-07.

Epoch 9: val_classification_output_accuracy did not improve from 0.77163
353/353 [==============================] - 77s 219ms/step - loss: 2.1076 - classification_output_loss: 0.1029 - classification_output_accuracy: 0.9688 - val_loss: 2.8585 - val_classification_output_loss: 0.8552 - val_classification_output_accuracy: 0.7631 - lr: 5.0000e-07
Epoch 10/15
353/353 [==============================] - ETA: 0s - loss: 2.0986 - classification_output_loss: 0.0960 - classification_output_accuracy: 0.9738
Epoch 10: val_classification_output_accuracy did not improve from 0.77163
353/353 [==============================] - 77s 219ms/step - loss: 2.0986 - classification_output_loss: 0.0960 - classification_output_accuracy: 0.9738 - val_loss: 2.8775 - val_classification_output_loss: 0.8756 - val_classification_output_accuracy: 0.7674 - lr: 2.5000e-07
Epoch 11/15
353/353 [==============================] - ETA: 0s - loss: 2.0972 - classification_output_loss: 0.0962 - classification_output_accuracy: 0.9702
Epoch 11: val_classification_output_accuracy did not improve from 0.77163
353/353 [==============================] - 82s 231ms/step - loss: 2.0972 - classification_output_loss: 0.0962 - classification_output_accuracy: 0.9702 - val_loss: 2.8796 - val_classification_output_loss: 0.8793 - val_classification_output_accuracy: 0.7617 - lr: 2.5000e-07

FINISHED FOLD 4
Peak Validation Accuracy: 0.7716